# Step 15 — A nonlinear order parameter: the census and the fidelity test on a pretrained CIFAR-10 model

**Order parameter.** For a class pair $(a,b)$, train a noise-conditional classifier $q_\sigma(a\mid x)$ on forward-noised images and set $\phi_\sigma(x)=2q_\sigma(a\mid x)-1\in[-1,1]$ (posterior magnetisation).

**Exact reference (no score needed).** $m=\phi_\sigma(x_0+\sigma\varepsilon)$ over the training data; its marginal score is estimated from a kernel density of the classifier **logit** (unbounded, smooth) and transformed to $m$, which avoids the edge bias of smoothing a bounded variable (validated to $\sim$3\% on the control). Stationary points of $\log p_\sigma(m)$ give the exact bifurcation diagram; the cusp fit gives $h$.

**Network side.** Mean-force identity for a nonlinear collective variable:
$$\frac{d}{dm}\log p_\sigma(m)=\mathbb E\Big[\frac{\nabla\phi\cdot s(x)}{|\nabla\phi|^2}+\nabla\cdot\frac{\nabla\phi}{|\nabla\phi|^2}\;\Big|\;m\Big],\qquad s=\nabla\log p_\sigma .$$
With $s$ from the pretrained EDM and the divergence term from the classifier (Hutchinson estimator with JVPs), the right-hand side is the network's implied marginal score along $\phi$; compare its zero crossings and its cusp coefficients to the exact ones.

**Validation first.** On the exactly solvable control with $\phi=\tanh(\cdot)$ (the true posterior magnetisation), both sides are known, so the divergence-corrected estimator is checked before it touches CIFAR-10.

GPU on, Internet on. ~45 min (classifiers ~2 min each; network sweeps dominate).

In [ ]:
import os, sys, subprocess, pickle, numpy as np, torch, torch.nn as nn, torch.nn.functional as F, math, time
import matplotlib.pyplot as plt
from torchvision import datasets
from scipy.signal import fftconvolve
dev = 'cuda' if torch.cuda.is_available() else 'cpu'; print(dev); rng = np.random.default_rng(0); torch.manual_seed(0); plt.rcParams['figure.dpi'] = 130

GRID = np.linspace(-0.998, 0.998, 3993); DG = GRID[1]-GRID[0]
def ref_score(z):
    """Exact marginal score of m = tanh(z/2) on GRID, from a Silverman-bandwidth KDE of the LOGIT z (unbounded, smooth).
    d/dm log p_m = (d/dz log p_z)/(dm/dz) + 2m/(1-m^2).  Validated to ~3% relative RMS at n=50k on the analytic control."""
    z = np.clip(z, -30, 30); bw = 1.5*1.06*z.std()*len(z)**(-1/5); G = np.linspace(z.min()-4*bw, z.max()+4*bw, 4001); dg = G[1]-G[0]
    h, _ = np.histogram(z, bins=np.append(G-dg/2, G[-1]+dg/2), density=True); k = np.exp(-G**2/(2*bw**2)); k /= k.sum(); pz = np.maximum(fftconvolve(h, k, 'same'), 1e-300)
    sz = np.gradient(np.log(pz), G); zq = 2*np.arctanh(GRID); return np.interp(zq, G, sz)/(0.5*(1-GRID**2)) + 2*GRID/(1-GRID**2)
def _merge(pts, merge=0.02):
    ch = True
    while ch and len(pts) > 1:
        ch = False
        for a in range(len(pts)-1):
            if abs(pts[a][0]-pts[a+1][0]) < merge and pts[a][1] != pts[a+1][1]: del pts[a:a+2]; ch = True; break
    return pts
def stationary_points(sm, lo=-0.997, hi=0.997, prominence=0.05):
    """Zero crossings of a marginal score sm on GRID; adjacent max/min pairs whose log-density differs by < prominence nats are dropped."""
    keep = (GRID > lo) & (GRID < hi); idx = np.where((np.sign(sm[:-1]) != np.sign(sm[1:])) & keep[:-1] & keep[1:])[0]
    pts = _merge([(float(GRID[i]-sm[i]*DG/(sm[i+1]-sm[i])) if sm[i+1] != sm[i] else float(GRID[i]), 'max' if sm[i] > 0 else 'min') for i in idx], merge=0.03)
    logp = np.concatenate([[0.0], np.cumsum(0.5*(sm[1:]+sm[:-1])*DG)]); lp = lambda m: float(np.interp(m, GRID, logp))
    ch = True
    while ch and len(pts) > 1:
        ch = False
        for a in range(len(pts)-1):
            if pts[a][1] != pts[a+1][1] and abs(lp(pts[a][0]) - lp(pts[a+1][0])) < prominence: del pts[a:a+2]; ch = True; break
    return pts
def n_max(pts): return sum(k == 'max' for _, k in pts)
def zero_crossings(cen, mean, cnt, min_count=40):
    okk = cnt >= min_count; c, f = cen[okk], mean[okk]; out = []
    for i in range(len(c)-1):
        if np.sign(f[i]) != np.sign(f[i+1]) and f[i] != 0: out.append((float(c[i]-f[i]*(c[i+1]-c[i])/(f[i+1]-f[i])), 'max' if f[i] > 0 else 'min'))
    return _merge(out, merge=0.05)
def cusp_fit(m, f, w, window=0.6, b_min=0.02, s_max=0.4):
    ok = (np.abs(m) < window) & (w > 0)
    if ok.sum() < 8: return dict(h=np.nan, a=np.nan, b=np.nan, clamped=True)
    c3, c2, c1, c0 = np.polyfit(m[ok], f[ok], 3, w=np.sqrt(w[ok])); b = max(-c3, b_min); c3 = -b; s = float(np.clip(-c2/(3*c3), -s_max, s_max))
    return dict(h=c0 + c1*s + c2*s**2 + c3*s**3, a=c1 + 3*c3*s**2 + 2*c2*s, b=b, clamped=(-c3 <= b_min+1e-12))
def bisect_sigma(get_pts, sigmas, iters=8):
    """Largest sigma at which the exact marginal of the order parameter is bimodal (scan downward in sigma)."""
    ss = sorted(sigmas, reverse=True); hi = None
    for s in ss:
        pts = get_pts(s)
        if n_max(pts) >= 2 and hi is not None: lo = s; break
        if n_max(pts) == 1: hi = s
    else: return None
    for _ in range(iters):
        mid = np.sqrt(lo*hi)
        if n_max(get_pts(mid)) >= 2: lo = mid
        else: hi = mid
    pm = [p for p, k in get_pts(hi) if k == 'max']; prev = pm[0] if pm else 0.0; pl = get_pts(lo)
    mx = sorted(p for p, k in pl if k == 'max'); mn = [p for p, k in pl if k == 'min']
    return float(lo), float(abs((mn[0] if mn else 0.5*(mx[0]+mx[-1])) - prev))

def mean_force_nonlinear(phi_fn, score_fn, x, sigma, n_hutch=4, eps=1e-2):
    """Per-sample estimate of  grad(phi).s/|grad phi|^2 + div(grad phi/|grad phi|^2)  at x (batch), with score s = score_fn(x, sigma).
    Divergence via Hutchinson with finite-difference JVPs of F(x) = grad phi/|grad phi|^2."""
    x = x.detach().requires_grad_(True)
    def Fvec(xx):
        xx = xx.detach().requires_grad_(True); ph = phi_fn(xx, sigma); g = torch.autograd.grad(ph.sum(), xx)[0]; return g/(g.pow(2).flatten(1).sum(1).view(-1, *[1]*(x.dim()-1)) + 1e-12)
    g = Fvec(x); s = score_fn(x, sigma).detach(); term1 = (g*s).flatten(1).sum(1)
    div = torch.zeros(len(x), device=x.device)
    for _ in range(n_hutch):
        v = torch.randn_like(x); v = v/v.flatten(1).norm(dim=1).view(-1, *[1]*(x.dim()-1))*math.sqrt(x[0].numel())
        Jv = (Fvec(x+eps*v) - Fvec(x-eps*v))/(2*eps); div += (Jv*v).flatten(1).sum(1)
    return (term1 + div/n_hutch).detach()

## A. Validation on the exactly solvable control with a nonlinear order parameter

Mixture with means $\pm s$, weight $w$; $\phi_\sigma(x)=\tanh(s\cdot x/\sigma^2+\eta)$ is the exact posterior magnetisation; the exact score and denoiser are analytic. The divergence-corrected estimator must reproduce the stationary points of the exact marginal of $\phi$.

In [ ]:
d = 256; s_pat = torch.from_numpy(np.sign(rng.standard_normal(d)).astype(np.float32)).to(dev); w = 0.7; eta = 0.5*math.log(w/(1-w))
def score_exact(x, sigma): return (torch.tanh(x @ s_pat/sigma**2 + eta)[:, None]*s_pat[None, :] - x)/sigma**2
def phi_exact(x, sigma): return torch.tanh(x @ s_pat/sigma**2 + eta)
def sample_ctrl(n, sigma):
    c = torch.rand(n, device=dev) < w; x0 = torch.where(c[:, None], s_pat[None, :], -s_pat[None, :]); return x0 + sigma*torch.randn(n, d, device=dev)

print(f"{'sigma':>7} {'exact stationary points of p(phi)':>36}   {'estimator zero crossings':>28}   {'RMS(est - exact) on |m|<0.7':>28}")
for sigma in [8.0, 12.0, 16.0, 20.0, 24.0]:
    x = sample_ctrl(60000, sigma); m = phi_exact(x, sigma).cpu().numpy(); z = 2*(x @ s_pat/sigma**2 + eta).cpu().numpy(); sm_ref = ref_score(z); ex = stationary_points(sm_ref)
    mf = torch.cat([mean_force_nonlinear(phi_exact, score_exact, xb, sigma) for xb in x.split(4000)]).cpu().numpy()
    edges = np.linspace(-0.99, 0.99, 60); b = np.clip(np.digitize(m, edges)-1, 0, 58); cnt = np.bincount(b, minlength=59).astype(float)
    est = np.bincount(b, weights=mf, minlength=59)/np.maximum(cnt, 1); cen = 0.5*(edges[:-1]+edges[1:])
    exact_curve = np.interp(cen, GRID, sm_ref); ok = (np.abs(cen) < 0.7) & (cnt >= 40)
    print(f"{sigma:7.1f} {str([(round(a_,3),k) for a_,k in ex]):>36}   {str([(round(a_,3),k) for a_,k in zero_crossings(cen, est, cnt)]):>28}   {np.sqrt(np.mean((est[ok]-exact_curve[ok])**2))/np.sqrt(np.mean(exact_curve[ok]**2)):28.3f}")
print("(tau = sigma^2/d: sigma=16 is tau=1. The reference itself is accurate to ~3%; relative RMS of ~0.05 certifies the estimator.)")

## B. CIFAR-10: noise-conditional classifiers, exact census along $\phi$, and pretrained-EDM fidelity

In [ ]:
if not os.path.exists('/kaggle/working/edm'): subprocess.run(['git', 'clone', '-q', 'https://github.com/NVlabs/edm.git', '/kaggle/working/edm'], check=True)
sys.path.insert(0, '/kaggle/working/edm'); import dnnlib
ds = datasets.CIFAR10('/kaggle/working/cifar', train=True, download=True)
X = torch.from_numpy((np.asarray(ds.data).astype(np.float32)/127.5 - 1.0).transpose(0, 3, 1, 2)); Y = np.asarray(ds.targets)
CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
with dnnlib.util.open_url('https://nvlabs-fi-cdn.nvidia.com/edm/pretrained/edm-cifar10-32x32-uncond-vp.pkl') as f: net = pickle.load(f)['ema'].to(dev).eval()
def score_net(x, sigma):
    with torch.no_grad(): D = net(x, torch.full((len(x),), float(sigma), device=dev))
    return (D - x)/sigma**2

class NoisyClf(nn.Module):
    def __init__(self, ch=48):
        super().__init__(); self.c1 = nn.Conv2d(3, ch, 3, padding=1); self.c2 = nn.Conv2d(ch, 2*ch, 3, padding=1); self.c3 = nn.Conv2d(2*ch, 4*ch, 3, padding=1)
        self.emb = nn.Linear(1, 4*ch); self.fc = nn.Linear(4*ch, 1)
    def forward(self, x, sigma):
        h = F.silu(self.c1(x)); h = F.max_pool2d(h, 2); h = F.silu(self.c2(h)); h = F.max_pool2d(h, 2); h = F.silu(self.c3(h)); h = h.mean((2, 3))
        h = h*(1 + self.emb(torch.log(sigma).view(-1, 1)/4)); return self.fc(h).squeeze(1)      # logit of class a
def train_clf(pair, epochs=6, bs=256):
    idx = np.where(np.isin(Y, pair))[0]; Xp = X[idx].to(dev); Yp = torch.from_numpy((Y[idx] == pair[0]).astype(np.float32)).to(dev)
    clf = NoisyClf().to(dev); opt = torch.optim.Adam(clf.parameters(), 1e-3); t0 = time.time()
    for ep in range(epochs):
        perm = torch.randperm(len(Xp), device=dev)
        for i in range(0, len(Xp)-bs+1, bs):
            j = perm[i:i+bs]; sigma = torch.exp(torch.empty(bs, device=dev).uniform_(math.log(0.05), math.log(30.0)))
            xt = Xp[j] + sigma.view(-1, 1, 1, 1)*torch.randn_like(Xp[j]); loss = F.binary_cross_entropy_with_logits(clf(xt, sigma), Yp[j]); opt.zero_grad(); loss.backward(); opt.step()
    clf.eval(); print(f"  classifier {pair} trained ({time.time()-t0:.0f}s)"); return clf, Xp

SIGMAS = np.geomspace(0.1, 30.0, 30)
def exact_census(clf, Xp, sigmas=SIGMAS, n=50000):
    """Exact marginal score of phi over forward-noised pair data (logit-space KDE, n=50k); stationary points; bisected transition; cusp fit."""
    def ref_at(sigma):
        with torch.no_grad():
            z = []
            for _ in range(n//10000):
                xb = Xp[torch.randint(len(Xp), (10000,), device=dev)]; xt = xb + sigma*torch.randn_like(xb); z.append(clf(xt, torch.full((10000,), float(sigma), device=dev)).cpu().numpy())
        z = np.concatenate(z); sm = ref_score(z); return sm, np.tanh(z/2)
    pts_at = lambda sigma: (stationary_points(ref_at(sigma)[0]), None)
    bj = bisect_sigma(lambda s: pts_at(s)[0], sigmas)
    if bj is None: return None
    sm, m = ref_at(bj[0]); wgt = np.histogram(m, bins=np.append(GRID-DG/2, GRID[-1]+DG/2))[0].astype(float); f = cusp_fit(GRID, sm, wgt)
    return dict(sigma_bif=bj[0], jump=bj[1], **f, pts_at=pts_at, ref_at=ref_at)

PAIRS = [(0, 8), (1, 9), (3, 5), (0, 1), (2, 4), (6, 7), (4, 7), (5, 7)]     # airplane/ship, auto/truck, cat/dog, airplane/auto, bird/deer, frog/horse, deer/horse, dog/horse
CENSUS = {}
print(f"{'pair':>22} {'sigma_bif (exact)':>18} {'jump':>6} {'h':>8} {'a':>8} {'b':>7}")
for pair in PAIRS:
    clf, Xp = train_clf(pair); r = exact_census(clf, Xp)
    if r is None: print(f"{CLASSES[pair[0]]+'/'+CLASSES[pair[1]]:>22}: never bimodal along phi"); continue
    CENSUS[pair] = (clf, Xp, r); print(f"{CLASSES[pair[0]]+'/'+CLASSES[pair[1]]:>22} {r['sigma_bif']:18.3f} {r['jump']:6.2f} {r['h']:+8.4f} {r['a']:+8.3f} {r['b']:7.3f}{'  [b clamped]' if r['clamped'] else ''}")

In [ ]:
# Fidelity of the pretrained unconditional EDM along phi, for each pair with a transition.
# The network's implied marginal score along phi = E[ grad(phi).s_net/|grad phi|^2 + div(grad phi/|grad phi|^2) | m ], binned.
# Its zero crossings are compared with the exact stationary points; the drift RMS error is reported at sigma_bif and 2 sigma_bif.
def net_curve(clf, Xp, sigma, n=6000, nbins=50):
    xb = Xp[torch.randperm(len(Xp), device=dev)[:n]]; xt = xb + sigma*torch.randn_like(xb)
    phi = lambda x, s: 2*torch.sigmoid(clf(x, torch.full((len(x),), float(s), device=dev)))-1
    with torch.no_grad(): m = phi(xt, sigma).cpu().numpy()
    mf = torch.cat([mean_force_nonlinear(phi, score_net, xc, sigma) for xc in xt.split(500)]).cpu().numpy()
    edges = np.linspace(-0.99, 0.99, nbins+1); b = np.clip(np.digitize(m, edges)-1, 0, nbins-1); cnt = np.bincount(b, minlength=nbins).astype(float)
    k = np.exp(-0.5*(np.arange(-3, 4)/1.2)**2); est = np.convolve(np.bincount(b, weights=mf, minlength=nbins), k, 'same')/np.maximum(np.convolve(cnt, k, 'same'), 1e-9)
    return 0.5*(edges[:-1]+edges[1:]), est, cnt, m

print(f"{'pair':>22} {'exact sigma_bif':>16} {'net sigma_bif':>14} {'exact h':>9} {'net h':>8} {'drift RMS @bif':>15} {'@2x':>7}")
fig, axes = plt.subplots(1, max(1, len(CENSUS)), figsize=(3.2*max(1, len(CENSUS)), 3.2)); axes = np.atleast_1d(axes)
for ax, (pair, (clf, Xp, r)) in zip(axes, CENSUS.items()):
    t0 = time.time()
    bjn = bisect_sigma(lambda s: zero_crossings(*net_curve(clf, Xp, s)[:3]), np.geomspace(0.3, 30.0, 16), iters=5)
    errs, hn = [], []
    for sg in [r['sigma_bif'], 2*r['sigma_bif']]:
        cen, est, cnt, m = net_curve(clf, Xp, sg); ex = np.interp(cen, GRID, r['ref_at'](sg)[0]); ok = (np.abs(cen) < 0.7) & (cnt >= 40)
        errs.append(float(np.sqrt(np.mean((est[ok]-ex[ok])**2))/np.sqrt(np.mean(ex[ok]**2)))); hn.append(cusp_fit(cen, est, cnt)['h'])
    print(f"{CLASSES[pair[0]]+'/'+CLASSES[pair[1]]:>22} {r['sigma_bif']:16.3f} {(bjn[0] if bjn else float('nan')):14.3f} {r['h']:+9.4f} {hn[0]:+8.4f} {errs[0]:15.3f} {errs[1]:7.3f}   ({time.time()-t0:.0f}s)")
    for sg in np.geomspace(0.3, 30.0, 16):
        for p_, k in r['pts_at'](sg)[0]: ax.plot(sg, p_, '_', ms=8, color='k', alpha=0.4)
        for p_, k in zero_crossings(*net_curve(clf, Xp, sg, n=3000)[:3]): ax.plot(sg, p_, 'o', ms=3, color='C0' if k == 'max' else 'C3')
    ax.set_xscale('log'); ax.set_xlabel('sigma'); ax.set_title(f"{CLASSES[pair[0]]}/{CLASSES[pair[1]]}", fontsize=9); ax.axhline(0, color='k', lw=0.4); ax.set_ylim(-1, 1)
plt.tight_layout(); plt.show()

## What to report back
**A.** The control table: estimator zero crossings vs exact, and relative RMS (should be ≲0.05). This certifies the divergence-corrected estimator.
**B.** The census table (exact, along the posterior magnetisation) for the eight CIFAR-10 pairs, and the fidelity table + diagrams for the pretrained EDM. This is the natural-image result the paper has been missing: a census of $h$ on CIFAR-10 and a learned-vs-exact comparison on a 56M-parameter model.